In [16]:
import os
import cv2 as cv
import numpy as np
from dataclasses import dataclass
from typing import List
#import pyautogui

In [17]:
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv.imread(os.path.join(folder,filename))
        if img is not None:
            images.append(img)
    return images

In [26]:
@dataclass
class Crop:
    tl_x: int
    tl_y: int
    br_x: int
    br_y: int


class ImageCrops:
    def __init__(self, base_name, train_image_path, validation_image_path, train_lbp_image_path, validation_lbp_image_path):
        self.base_name = base_name
        self.train_image_path = train_image_path
        self.validation_image_path = validation_image_path
        self.train_ldb_image_path = train_lbp_image_path
        self.validation_ldb_image_path = validation_lbp_image_path
        self.all_areas: List[Crop] = []
    #base_image: cv.Mat
    #base_image_root: str
    #all_areas: List[Crop]
    #base_name: str
    #number_of_crops : int


In [27]:
'''
path_to_folder = "../DB/images/textures/IMAGES/base/"
texture_path = "../DB/images/textures/IMAGES/texturas/"
'''
path_to_folder = "../DB/images/texturas-nuevas/class/"
texture_path = "../DB/images/texturas-nuevas/class/"

if not os.path.exists(texture_path):
            os.mkdir(texture_path)
train_path = "../DB/images/texturas-nuevas/class/"
if not os.path.exists(train_path):
            os.mkdir(texture_path+"train/")
validation_path = "../DB/images/texturas-nuevas/class/"
if not os.path.exists(validation_path):
            os.mkdir(texture_path+"validation/")

## LBP feature patterns folders
lbp_path = "../DB/images/texturas-nuevas/lbp/"
if not os.path.exists(lbp_path):
    os.mkdir(lbp_path)

train_lbp_path = "../DB/images/texturas-nuevas/lbp/train/"
if not os.path.exists(train_lbp_path):
    os.mkdir(train_lbp_path)

validation_lbp_path = "../DB/images/texturas-nuevas/lbp/validation/"
if not os.path.exists(validation_lbp_path):
    os.mkdir(validation_lbp_path)

all_image_subfolder_names = os.listdir(path_to_folder)
all_image_subfolder_names.sort()
all_analysis = []
###
for texture_class_name in all_image_subfolder_names:
    print(texture_class_name)
    #all_image_names = os.listdir(path_to_folder+all_image_subfolder_names+texture_class_name)
    # all_images = load_images_from_folder(path_to_folder+texture_class_name)
    all_images =[]
    #name_without_extension = texture_class_name.split(".")
    train_folder_path = train_path+texture_class_name
    validation_folder_path = validation_path+texture_class_name

    train_lbp_folder_path = train_lbp_path+texture_class_name
    validation_lbp_folder_path = validation_lbp_path+texture_class_name

    if not os.path.exists(train_folder_path):
            os.mkdir(train_folder_path)

    if not os.path.exists(validation_folder_path):
        os.mkdir(validation_folder_path)

    if not os.path.exists(train_lbp_folder_path):
        os.mkdir(train_lbp_folder_path)

    if not os.path.exists(validation_lbp_folder_path):
        os.mkdir(validation_lbp_folder_path)

    image_folder_path = os.path.join(path_to_folder, texture_class_name)
    all_images = load_images_from_folder(image_folder_path)

    for img in all_images:
        imagCrops = ImageCrops(texture_class_name, train_folder_path, validation_folder_path, train_lbp_folder_path, validation_lbp_folder_path)
        imagCrops.base_image = img
        height, width = img.shape[:2]
        imagCrops.base_image_width = width
        imagCrops.base_image_height = height
        all_analysis.append(imagCrops)

print("Number of classes: "+str(len(all_image_subfolder_names)))
"Number of images to analysis: "+str(len(all_analysis))

S00
S01
S02
S03
S04
S05
S06
S07
Number of classes: 8


'Number of images to analysis: 717'

Subdivision function

In [28]:
def subdivide_area(area: Crop, size_of_subpatches: int):
    list_of_areas = []

    area_width = area.br_x - area.tl_x
    area_height = area.br_y - area.tl_y

    num_horizontal = round(area_width/size_of_subpatches)
    num_vertical = round(area_height/size_of_subpatches)

    print("Number of subpatches: ", (num_vertical*num_horizontal))

    for y in range(num_vertical):
        for x in range(num_horizontal):
            s_tl_x = area.tl_x + x*size_of_subpatches
            s_tl_y = area.tl_y + y*size_of_subpatches
            s_br_x = area.tl_x + (x*size_of_subpatches)+size_of_subpatches
            s_br_y = area.tl_y + (y*size_of_subpatches)+size_of_subpatches

            sub_patch = Crop(tl_x=s_tl_x, tl_y=s_tl_y, br_x=s_br_x, br_y=s_br_y)
            list_of_areas.append(sub_patch)


    return list_of_areas


Mouse Callback function

In [31]:
# mouse callback function
def draw_rectanlge(event, x, y, flags, param):
    """ Draw rectangle on mouse click and drag """
    global drawing, ix,iy,bx,by, crop_area

    # if the left mouse button was clicked, record the starting and set the drawing flag to True
    if event == cv.EVENT_LBUTTONDOWN:
        drawing = True
        ix,iy = x,y
    # mouse is being moved, draw rectangle
    elif event == cv.EVENT_MOUSEMOVE:
        if drawing:
            bx, by = x, y

    # if the left mouse button was released, set the drawing flag to False
    elif event == cv.EVENT_LBUTTONUP:
        bx,by = x,y
        crop_area = Crop(tl_x=ix, tl_y=iy, br_x=bx, br_y=by)
        drawing = False



Select interest area

In [32]:

for single_analysis in all_analysis:

    drawing = False # true if mouse is pressed
    rendering = True
    ix,iy = -1,-1
    bx,by = -1, -1
    crop_area = Crop(tl_x=0, tl_y=0, br_x=0, br_y=0)


    img_orig = single_analysis.base_image.copy()
    img_draw = single_analysis.base_image.copy()

    #cv.namedWindow(single_analysis.base_name,cv.WINDOW_NORMAL)
    #cv.namedWindow(single_analysis.base_name,cv.WINDOW_AUTOSIZE)
    cv.namedWindow(single_analysis.base_name,cv.WINDOW_NORMAL | cv.WINDOW_KEEPRATIO)
    cv.setMouseCallback(single_analysis.base_name,draw_rectanlge)


    while rendering:

        img_draw = img_orig.copy()
        cv.rectangle(img_draw, (ix, iy), (bx, by), (255, 0, 0), 4)
        cv.circle(img_draw, (ix, iy), 10, (0, 0, 255), -1)
        cv.circle(img_draw, (bx, by), 10, (0, 0, 255), -1)

        cv.resizeWindow(single_analysis.base_name, (1280,960))
        cv.imshow(single_analysis.base_name,img_draw)

        if cv.waitKey(20) & 0xFF == 27:
            break

    cv.destroyAllWindows()

    single_analysis.crop_area = crop_area


In [34]:
size_of_subpatches = 128 # in pixels

for single_analysis in all_analysis:

    areas = [] # array of Crops
    areas = subdivide_area(single_analysis.crop_area, size_of_subpatches)
    single_analysis.all_areas = areas
    single_analysis.number_of_crops = len(areas)

    "In "+single_analysis.base_name+" was created "+str(len(areas))+" patches"

Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of subpatches:  88
Number of su

In [35]:
for single_analysis in all_analysis:
    height, width = single_analysis.base_image.shape[:2]
    crop_area = Crop(tl_x=0, tl_y=0, br_x=width, br_y=height)
    single_analysis.crop_area = crop_area

Save patches function

In [36]:
from skimage import feature

class LocalBinaryPatterns:
	def __init__(self, numPoints, radius):
		# store the number of points and radius
		self.numPoints = numPoints
		self.radius = radius
	def describe(self, image, eps=1e-7):
		# compute the Local Binary Pattern representation
		# of the image, and then use the LBP representation
		lbp = feature.local_binary_pattern(image, self.numPoints,
			self.radius, method="uniform")
		return lbp

In [37]:
def convert_lbp(image):
  desc = LocalBinaryPatterns(24, 8)
  data = []

  # load the image, convert it to grayscale, and describe it
  gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
  mat = desc.describe(gray)
  # extract the label from the image path, then update the
  # label and data lists

  # data = numpy.resize(data,(32,32))
  dst = np.dstack([mat, mat, mat])
  data.append(dst)
  data = np.array(data)
  print(data.shape)
  return data

In [38]:

def save_cropped_images(current_image, all_areas, train_path, validation_path, train_lbp_path, validation_lbp_path, filesname):

   # cv.imshow(filesname,current_image)
    # cv.waitKey()

    train_path = train_path
    validation_path = validation_path

    saving_path = train_path
    ldb_saving_path = train_lbp_path

    if not os.path.exists(train_path):
        os.mkdir(train_path)

    if not os.path.exists(validation_path):
        os.mkdir(validation_path)

    if not os.path.exists(train_lbp_path):
        os.mkdir(train_lbp_path)

    if not os.path.exists(validation_lbp_path):
        os.mkdir(validation_lbp_path)

    iter = 0
    ## loop over every bounding box save in array "ROIs"

    number_of_test = round(len(all_areas)*0.8)
    print(number_of_test)
    for i in range(len(all_areas)):
        #crop roi from original image
        to_crop = current_image.copy()
        crop = all_areas[i]

        if i >= number_of_test:
            saving_path  = validation_path
            ldb_saving_path = validation_lbp_path

        # print("Save path: ",crop.tl_x,", ", crop.br_x,", ",crop.tl_y,", ", crop.br_y)
        img_crop=to_crop[crop.tl_y:crop.br_y,crop.tl_x:crop.br_x]
        #cv.imshow('crop', img_crop)

        while os.path.exists(saving_path +"/"+ str(filesname)+"."+str(iter)+".jpeg"):
            iter+=1


        #save cropped image
        cv.imwrite(saving_path +"/"+ str(filesname)+"."+str(iter)+".jpeg",img_crop)

        lbp_train_image = convert_lbp(img_crop)
        cv.imwrite(ldb_saving_path +"/"+ str(filesname)+"."+str(iter)+".jpeg", lbp_train_image[0])

        iter+=1



Saving patches in folder

In [39]:
for single_analysis in all_analysis:
    save_cropped_images(single_analysis.base_image, single_analysis.all_areas, single_analysis.train_image_path, single_analysis.validation_image_path, single_analysis.train_ldb_image_path, single_analysis.validation_ldb_image_path, single_analysis.base_name)

70
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 128, 3)
(1, 128, 12